# Ensemble Learning for Myocardial Perfusion Imaging Classification

## Overview
This notebook implements six ensemble combination rules applied to pre-trained transfer learning models for binary classification of myocardial perfusion SPECT images (ischemic vs. non-ischemic). It was developed as part of a benchmarking study comparing transfer learning and ensemble learning approaches for computer-aided diagnosis in nuclear cardiology.

## How it works
Each constituent model was trained independently for 100 runs with different random initialisations (see `TL_fixedvalidation.py`). The predicted class-1 probabilities from all runs are stored in an Excel file. This notebook loads those predictions and applies six combination rules — **Majority Voting, Sum Rule, Weighted Sum Rule, Borda Count, Max Rule, and Median Rule** — to Top-3 and Top-5 model pools (12 configurations total), evaluated run-by-run across all 100 runs.

## Inputs
- `metrics.xlsx` — Excel file with columns: `model_name`, `predicts` (comma-separated class-1 probabilities for each test image), `tag` (run identifier). Place in `../data/` or update `DATA_DIR`.
- `test_labels.txt` — Plain text file with one true binary label (0 or 1) per line, one per test image.

## Outputs
- Median and IQR of ACC, PRE, SEN, SPE, F1S, and AUC across 100 runs for each of the 12 ensemble configurations.
- Optional: per-run results exported to Excel via `export_results()`.

In [ ]:
# Cell 1: Imports and configuration
import pandas as pd
import numpy as np
import os
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

# ── User configuration ──────────────────────────────────────────────────────
DATA_DIR     = '../data'          # Directory containing input files
EXCEL_FILE   = 'metrics.xlsx'     # Model predictions file
LABELS_FILE  = 'test_labels.txt'  # True test-set labels
NUM_RUNS     = 100                # Number of independent training runs per model
# ─────────────────────────────────────────────────────────────────────────────

excel_path  = os.path.join(DATA_DIR, EXCEL_FILE)
labels_path = os.path.join(DATA_DIR, LABELS_FILE)

# Load model predictions
df_models = pd.read_excel(excel_path)

# Load true labels
with open(labels_path, 'r') as f:
    true_labels = np.array([int(line.strip()) for line in f if line.strip()])

N_TEST = len(true_labels)  # Number of test images

print(f"Loaded {len(df_models)} model-run records")
print(f"Test set: {N_TEST} images | "
      f"{np.sum(true_labels)} ischemic (class 1), "
      f"{N_TEST - np.sum(true_labels)} non-ischemic (class 0)")
print(f"Models found: {sorted(df_models['model_name'].unique())}")

In [ ]:
# Cell 2: Define model pools
# Update these lists to match the model names in your Excel file.
# The pools below reflect the Top-3 and Top-5 models selected by median AUC
# in the original study; replace with your own ranked model names as needed.

TOP3_MODELS = [
    'InceptionResNetV2',
    'DenseNet201',
    'Xception'
]

TOP5_MODELS = [
    'InceptionResNetV2',
    'DenseNet201',
    'Xception',
    'InceptionV3',
    'ResNet50V2'
]

POOLS = {'Top-3': TOP3_MODELS, 'Top-5': TOP5_MODELS}

In [ ]:
# Cell 3: Helper functions

def parse_predictions(pred_string):
    """Parse a comma- or space-separated string of probabilities into a numpy array."""
    if isinstance(pred_string, (list, np.ndarray)):
        return np.array(pred_string, dtype=float)
    cleaned = str(pred_string).strip('[](){} ')
    sep = ',' if ',' in cleaned else None
    return np.array([float(x) for x in cleaned.split(sep) if x.strip()])


def get_run_predictions(model_names, run_idx):
    """
    Retrieve the class-1 probability vectors for a given run index (0-based)
    from each model in `model_names`.
    Returns a 2-D array of shape (K, N_TEST).
    """
    probs = []
    for name in model_names:
        rows = df_models[df_models['model_name'] == name].reset_index(drop=True)
        if run_idx >= len(rows):
            raise IndexError(f"Run {run_idx} not available for model '{name}' "
                             f"(only {len(rows)} runs found).")
        p = parse_predictions(rows.loc[run_idx, 'predicts'])[:N_TEST]
        probs.append(p)
    return np.vstack(probs)   # shape: (K, N_TEST)


def compute_metrics(y_true, y_pred, y_prob=None):
    """Return a dict of ACC, PRE, SEN, SPE, F1S, and AUC (if y_prob provided)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        'ACC': accuracy_score(y_true, y_pred),
        'PRE': precision_score(y_true, y_pred, zero_division=0),
        'SEN': recall_score(y_true, y_pred, zero_division=0),
        'SPE': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        'F1S': f1_score(y_true, y_pred, zero_division=0),
        'AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    }
    return metrics

In [ ]:
# Cell 4: Combination rule implementations

def majority_voting(probs):
    """
    Majority Voting: each model votes the class with highest probability
    (threshold 0.5); the class with the most votes wins.
    Returns (y_pred, y_prob=None).
    """
    votes = (probs >= 0.5).astype(int)          # shape: (K, N)
    y_pred = (votes.sum(axis=0) > probs.shape[0] / 2).astype(int)
    return y_pred, None


def sum_rule(probs):
    """
    Sum Rule (soft voting): average class-1 probabilities across models;
    classify as 1 if average >= 0.5.
    Returns (y_pred, y_prob).
    """
    y_prob = probs.mean(axis=0)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob


def weighted_sum_rule(probs, weights):
    """
    Weighted Sum Rule: weighted average of class-1 probabilities.
    `weights` is a 1-D array of length K (e.g., median AUC per model);
    it is normalised internally so it sums to 1.
    Returns (y_pred, y_prob).
    """
    w = np.array(weights, dtype=float)
    w /= w.sum()
    y_prob = (w[:, None] * probs).sum(axis=0)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob


def borda_count(probs):
    """
    Borda Count: each model ranks test images by ascending class-1 probability
    (rank 1 = lowest probability). Ranks are summed and normalised by the
    maximum possible Borda score (K * N), mapping scores to [1/N, 1].
    Images with a normalised score > 0.5 are classified as ischemic (class 1).
    Note: this threshold assumes approximately equal class prevalence.
    Returns (y_pred, y_prob) where y_prob is the normalised Borda score.
    """
    K, N = probs.shape
    ranks = np.argsort(np.argsort(probs, axis=1), axis=1) + 1  # 1-based ranks
    borda_scores = ranks.sum(axis=0)
    y_prob = borda_scores / (K * N)
    y_pred = (y_prob > 0.5).astype(int)
    return y_pred, y_prob


def max_rule(probs):
    """
    Max Rule: ensemble class-1 probability is the maximum across all models.
    Returns (y_pred, y_prob).
    """
    y_prob = probs.max(axis=0)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob


def median_rule(probs):
    """
    Median Rule: ensemble class-1 probability is the median across all models.
    Returns (y_pred, y_prob).
    """
    y_prob = np.median(probs, axis=0)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob


RULES = {
    'Majority Voting': majority_voting,
    'Sum Rule':        sum_rule,
    'Max Rule':        max_rule,
    'Median Rule':     median_rule,
    # Weighted Sum Rule and Borda Count are handled separately (see Cell 5)
}

In [ ]:
# Cell 5: Compute per-model AUC weights for Weighted Sum Rule
# Weights = median AUC across all runs for each model in the pool.

def get_model_auc_weights(model_names, n_runs):
    """Compute median AUC across runs for each model; return as normalised weight array."""
    aucs = []
    for name in model_names:
        rows = df_models[df_models['model_name'] == name].reset_index(drop=True)
        model_aucs = []
        for run_idx in range(min(n_runs, len(rows))):
            p = parse_predictions(rows.loc[run_idx, 'predicts'])[:N_TEST]
            try:
                auc = roc_auc_score(true_labels, p)
                model_aucs.append(auc)
            except Exception:
                pass
        aucs.append(np.median(model_aucs) if model_aucs else 0.5)
    weights = np.array(aucs)
    weights /= weights.sum()
    return weights

print("Computing AUC weights...")
auc_weights = {pool_name: get_model_auc_weights(models, NUM_RUNS)
               for pool_name, models in POOLS.items()}

for pool_name, models in POOLS.items():
    print(f"\n{pool_name} weights:")
    for m, w in zip(models, auc_weights[pool_name]):
        print(f"  {m}: {w:.4f}")

In [ ]:
# Cell 6: Run all 12 ensemble configurations (6 rules × 2 pools)

all_results = {}  # key: (pool_name, rule_name) → list of metric dicts (one per run)

for pool_name, models in POOLS.items():
    weights = auc_weights[pool_name]
    
    for run_idx in range(NUM_RUNS):
        probs = get_run_predictions(models, run_idx)  # shape: (K, N_TEST)
        
        # --- Four rules via RULES dict ---
        for rule_name, rule_fn in RULES.items():
            y_pred, y_prob = rule_fn(probs)
            m = compute_metrics(true_labels, y_pred, y_prob)
            key = (pool_name, rule_name)
            all_results.setdefault(key, []).append(m)
        
        # --- Weighted Sum Rule ---
        y_pred, y_prob = weighted_sum_rule(probs, weights)
        m = compute_metrics(true_labels, y_pred, y_prob)
        all_results.setdefault((pool_name, 'Weighted Sum Rule'), []).append(m)
        
        # --- Borda Count ---
        y_pred, y_prob = borda_count(probs)
        m = compute_metrics(true_labels, y_pred, y_prob)
        all_results.setdefault((pool_name, 'Borda Count'), []).append(m)

print(f"Completed {len(all_results)} configurations × {NUM_RUNS} runs each.")

In [ ]:
# Cell 7: Summarise results — Median and IQR across 100 runs

METRIC_COLS = ['ACC', 'PRE', 'SEN', 'SPE', 'F1S', 'AUC']
RULE_ORDER  = ['Majority Voting', 'Sum Rule', 'Weighted Sum Rule',
               'Borda Count', 'Max Rule', 'Median Rule']

summary_rows = []
for pool_name in ['Top-3', 'Top-5']:
    for rule_name in RULE_ORDER:
        key = (pool_name, rule_name)
        if key not in all_results:
            continue
        run_metrics = pd.DataFrame(all_results[key])  # shape: (100, 6)
        row = {'Pool': pool_name, 'Rule': rule_name}
        for col in METRIC_COLS:
            vals = run_metrics[col].dropna()
            row[f'{col}_Median'] = np.median(vals)
            row[f'{col}_IQR']    = np.percentile(vals, 75) - np.percentile(vals, 25)
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Display formatted
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(summary_df.to_string(index=False))

In [ ]:
# Cell 8: Export results to Excel (optional)
# Uncomment and run to save per-run results and summary to an Excel workbook.

# OUTPUT_FILE = 'ensemble_results.xlsx'
#
# with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
#     summary_df.to_excel(writer, sheet_name='Summary', index=False)
#     for (pool_name, rule_name), runs in all_results.items():
#         sheet_name = f"{pool_name}_{rule_name[:15]}".replace(' ', '_')
#         pd.DataFrame(runs).to_excel(writer, sheet_name=sheet_name, index=False)
#
# print(f"Results exported to {OUTPUT_FILE}")